# Day 4 — Feature Engineering & Ensembles
## RetailMind AI | Customer Churn Prediction

**Goal:** improve the Day 3 churn model using domain feature engineering, dimensionality reduction / feature selection, and ensemble models (Random Forest bagging and XGBoost boosting). Compare results fairly, select a production candidate, and save the complete preprocessing + model pipeline.

### Project conventions
- Keep the existing Day 1–Day 3 files unchanged.
- Expected project layout:

```text
RETAILMIND_AI/
├── data/
│   └── Telco-Customer-Churn.csv
├── models/
│   ├── churn_v3.joblib          # Day 3 champion (if available)
│   └── ...
├── notebooks/
│   └── Day_4_Feature_Engineering_Ensembles.ipynb
├── README.md
└── ...
```

Run this notebook from the `notebooks/` folder. It also checks the common alternative of running from the project root.

**Dataset:** Telco Customer Churn — Kaggle (`blastchar/telco-customer-churn`).  
**Libraries for this increment:** pandas, NumPy, scikit-learn, XGBoost, joblib, matplotlib, seaborn.

> The notebook does not overwrite the Day 3 model. The Day 4 candidate is saved separately as `models/churn_v4.joblib`.


## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

from xgboost import XGBClassifier

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")


## 2. Locate the project, dataset, and model folders

In [ ]:
# Works whether the notebook is launched from RETAILMIND_AI/ or RETAILMIND_AI/notebooks/
CWD = Path.cwd()
if CWD.name.lower() == "notebooks":
    PROJECT_DIR = CWD.parent
elif (CWD / "data").exists():
    PROJECT_DIR = CWD
elif (CWD.parent / "data").exists():
    PROJECT_DIR = CWD.parent
else:
    PROJECT_DIR = CWD

DATA_PATH = PROJECT_DIR / "data" / "Telco-Customer-Churn.csv"
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR.resolve())
print("Dataset:", DATA_PATH.resolve())
print("Dataset exists:", DATA_PATH.exists())


## 3. Load and inspect the Telco churn data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Place Telco-Customer-Churn.csv "
        "inside RETAILMIND_AI/data/ and rerun this cell."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.isna().sum().sort_values(ascending=False).head(10))


### Clean source data

In [ ]:
# TotalCharges is stored as text in the original Telco dataset; blank strings become NaN.
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove duplicate records if present.
df = df.drop_duplicates().copy()

TARGET = "Churn"
if TARGET not in df.columns:
    raise KeyError(f"Expected target column '{TARGET}' was not found.")

# Normalize target labels to 0/1.
df[TARGET] = df[TARGET].astype(str).str.strip().map({"No": 0, "Yes": 1})
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

print("Cleaned shape:", df.shape)
print("Churn distribution:")
display(df[TARGET].value_counts(normalize=True).rename("proportion"))


## 4. Domain feature engineering

Create customer-lifecycle and billing features that may help explain churn:
- **TenureGroup:** customer lifecycle bucket.
- **AvgMonthlySpend:** total charges divided by tenure, with a safe zero-tenure fallback.
- **MonthlyToTotalRatio:** monthly charge relative to accumulated charges.
- **HasPartner / HasDependents / HasPhone / HasInternet:** simple service and household flags.
- **NumServices:** count of subscribed add-on services where the source columns exist.

These are created from the raw columns before splitting. No target information is used.

In [ ]:
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()

    if "tenure" in data.columns:
        data["TenureGroup"] = pd.cut(
            data["tenure"],
            bins=[-1, 6, 12, 24, 48, 72],
            labels=["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"]
        ).astype("object")

    if {"TotalCharges", "tenure"}.issubset(data.columns):
        safe_tenure = data["tenure"].replace(0, np.nan)
        data["AvgMonthlySpend"] = (data["TotalCharges"] / safe_tenure).fillna(
            data["MonthlyCharges"] if "MonthlyCharges" in data.columns else 0
        )

    if {"MonthlyCharges", "TotalCharges"}.issubset(data.columns):
        data["MonthlyToTotalRatio"] = (
            data["MonthlyCharges"] / data["TotalCharges"].replace(0, np.nan)
        ).replace([np.inf, -np.inf], np.nan)

    flag_sources = {
        "HasPartner": "Partner",
        "HasDependents": "Dependents",
        "HasPhone": "PhoneService",
        "HasInternet": "InternetService",
    }
    for new_col, source_col in flag_sources.items():
        if source_col in data.columns:
            data[new_col] = (data[source_col].astype(str).str.lower() == "yes").astype(int)

    service_cols = [
        c for c in [
            "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
        ] if c in data.columns
    ]
    if service_cols:
        data["NumServices"] = sum(
            (data[c].astype(str).str.lower() == "yes").astype(int)
            for c in service_cols
        )

    return data

featured_df = engineer_features(df)
print("Before:", df.shape, "| After feature engineering:", featured_df.shape)
display(featured_df.head())


## 5. Define features and hold-out split

In [ ]:
X = featured_df.drop(columns=[TARGET])
y = featured_df[TARGET]

# CustomerID is an identifier, not a useful predictive feature.
X = X.drop(columns=["customerID"], errors="ignore")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


## 6. Preprocessing pipelines

Imputation, scaling, and one-hot encoding are fitted only on the training folds through scikit-learn pipelines. This prevents preprocessing leakage into validation/test data.

In [ ]:
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # compatibility with older scikit-learn versions
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


## 7. Day 3 champion baseline (if available)

The notebook tries to load `models/churn_v3.joblib`. If it is a complete fitted pipeline, its hold-out score is reported. If the saved object expects a different input format, the cell records that fact and continues; it does not alter the Day 3 artifact.

In [ ]:
DAY3_MODEL_PATH = MODELS_DIR / "churn_v3.joblib"
day3_baseline = None

if DAY3_MODEL_PATH.exists():
    try:
        day3_model = joblib.load(DAY3_MODEL_PATH)
        day3_pred = day3_model.predict(X_test)
        if hasattr(day3_model, "predict_proba"):
            day3_prob = day3_model.predict_proba(X_test)[:, 1]
            day3_auc = roc_auc_score(y_test, day3_prob)
        else:
            day3_auc = np.nan
        day3_baseline = {
            "Model": "Day 3 Champion",
            "Accuracy": accuracy_score(y_test, day3_pred),
            "Precision": precision_score(y_test, day3_pred, zero_division=0),
            "Recall": recall_score(y_test, day3_pred, zero_division=0),
            "F1": f1_score(y_test, day3_pred, zero_division=0),
            "ROC_AUC": day3_auc
        }
        print("Day 3 champion evaluated successfully.")
    except Exception as exc:
        print("Could not evaluate Day 3 artifact with this notebook's raw features.")
        print("Reason:", repr(exc))
        print("Continue with the Day 4 models; compare against the Day 3 metrics in README.")
else:
    print("models/churn_v3.joblib not found. Day 4 models will still be trained.")


## 8. Compare feature reduction approaches

We compare two approaches:
1. **PCA:** retain 95% of the variance after preprocessing.
2. **Model-based feature selection:** retain features selected by a Random Forest.

Both transformations are inside pipelines, so they are refitted inside each CV fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_selector = SelectFromModel(
    estimator=RandomForestClassifier(
        n_estimators=150, random_state=RANDOM_STATE,
        class_weight="balanced", n_jobs=-1
    ),
    threshold="median"
)

pca_rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(
        n_estimators=250, min_samples_leaf=2,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ))
])

selected_rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("select", rf_selector),
    ("model", RandomForestClassifier(
        n_estimators=250, min_samples_leaf=2,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ))
])

reduction_results = []
for name, estimator in [
    ("Random Forest + PCA", pca_rf_pipeline),
    ("Random Forest + feature selection", selected_rf_pipeline)
]:
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv,
        scoring={"roc_auc": "roc_auc", "f1": "f1"},
        n_jobs=-1, return_train_score=True
    )
    reduction_results.append({
        "Approach": name,
        "Train ROC-AUC": scores["train_roc_auc"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
        "CV F1": scores["test_f1"].mean()
    })

reduction_df = pd.DataFrame(reduction_results).sort_values("CV ROC-AUC", ascending=False)
display(reduction_df.round(4))


## 9. Train bagging and boosting ensembles

- **Bagging:** Random Forest trains many decision trees on bootstrapped samples and combines their predictions.
- **Boosting:** XGBoost builds trees sequentially, focusing on errors made by earlier trees.

We use class weighting for the churn class imbalance and compare cross-validation ROC-AUC, train ROC-AUC, and F1.

In [ ]:
# Estimate the negative/positive class ratio for XGBoost.
positive_count = int((y_train == 1).sum())
negative_count = int((y_train == 0).sum())
scale_pos_weight = negative_count / max(positive_count, 1)

rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=350,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=350,
        max_depth=3,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2,
        min_child_weight=2,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

candidate_models = {
    "Random Forest (Bagging)": rf_pipeline,
    "XGBoost (Boosting)": xgb_pipeline
}

ensemble_results = []
for name, estimator in candidate_models.items():
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv,
        scoring={"roc_auc": "roc_auc", "f1": "f1"},
        n_jobs=-1, return_train_score=True
    )
    ensemble_results.append({
        "Model": name,
        "Train ROC-AUC": scores["train_roc_auc"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "ROC-AUC Std": scores["test_roc_auc"].std()
    })

ensemble_df = pd.DataFrame(ensemble_results).sort_values("CV ROC-AUC", ascending=False)
display(ensemble_df.round(4))


## 10. Select the Day 4 candidate and evaluate once on the test set

Selection is based on mean 5-fold training CV ROC-AUC, not on the test set. The hold-out test set is used only for the final unbiased evaluation.

In [ ]:
best_model_name = ensemble_df.iloc[0]["Model"]
best_pipeline = candidate_models[best_model_name]
best_pipeline.fit(X_train, y_train)

test_pred = best_pipeline.predict(X_test)
test_prob = best_pipeline.predict_proba(X_test)[:, 1]

day4_metrics = {
    "Model": best_model_name,
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "F1": f1_score(y_test, test_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, test_prob)
}

comparison_rows = []
if day3_baseline is not None:
    comparison_rows.append(day3_baseline)
comparison_rows.append(day4_metrics)
final_comparison = pd.DataFrame(comparison_rows)
display(final_comparison.round(4))

print("Selected model:", best_model_name)
print("\nClassification report:")
print(classification_report(y_test, test_pred, target_names=["No Churn", "Churn"], zero_division=0))


### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted No", "Predicted Yes"],
    yticklabels=["Actual No", "Actual Yes"]
)
plt.title(f"Day 4 — {best_model_name}")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## 11. Bias–variance check

A large gap between training and CV ROC-AUC can indicate overfitting (high variance). Low scores on both can indicate underfitting (high bias). Use the table below alongside the learning curve; do not select a model using training score alone.

In [ ]:
display(ensemble_df[["Model", "Train ROC-AUC", "CV ROC-AUC", "ROC-AUC Std"]].round(4))

gap_df = ensemble_df.copy()
gap_df["Train-CV Gap"] = gap_df["Train ROC-AUC"] - gap_df["CV ROC-AUC"]
display(gap_df[["Model", "Train-CV Gap", "ROC-AUC Std"]].round(4))


### Learning curve for the selected model

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, validation_scores = learning_curve(
    best_pipeline, X_train, y_train,
    cv=cv, scoring="roc_auc",
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=-1
)

plt.figure(figsize=(7, 4))
plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training ROC-AUC")
plt.plot(train_sizes, validation_scores.mean(axis=1), marker="o", label="CV ROC-AUC")
plt.xlabel("Number of training examples")
plt.ylabel("ROC-AUC")
plt.title("Learning Curve — Day 4 Selected Model")
plt.legend()
plt.tight_layout()
plt.show()


## 12. Persist the production pipeline and evaluation metrics

Save the **entire fitted pipeline**, including feature preprocessing, so inference uses exactly the same transformations as training. The Day 3 model remains untouched.

In [ ]:
MODEL_PATH = MODELS_DIR / "churn_v4.joblib"
METRICS_PATH = MODELS_DIR / "churn_v4_metrics.csv"

artifact = {
    "pipeline": best_pipeline,
    "model_name": best_model_name,
    "feature_columns": X.columns.tolist(),
    "target": TARGET,
    "metrics": day4_metrics,
    "random_state": RANDOM_STATE
}
joblib.dump(artifact, MODEL_PATH)
pd.DataFrame([day4_metrics]).to_csv(METRICS_PATH, index=False)

print("Saved model:", MODEL_PATH.resolve())
print("Saved metrics:", METRICS_PATH.resolve())


## 13. Verify that the saved model can be loaded

In [ ]:
loaded_artifact = joblib.load(MODEL_PATH)
loaded_pipeline = loaded_artifact["pipeline"]

sample_predictions = loaded_pipeline.predict(X_test.head(5))
print("Reload verification passed.")
print("Sample predictions (0 = No churn, 1 = Churn):", sample_predictions.tolist())


## 14. Update the README baseline table

This cell creates a Markdown summary and safely replaces only the Day 4 section between markers. Other README content is preserved. Review the generated section and commit it with the model/metrics artifact.

In [ ]:
README_PATH = PROJECT_DIR / "README.md"

def markdown_table(dataframe):
    return dataframe.to_markdown(index=False)

baseline_table = final_comparison.copy()
baseline_table = baseline_table.round(4)

day4_readme_section = (
    "<!-- DAY4_RESULTS_START -->\n"
    "## Day 4 — Feature Engineering & Ensembles Results\n\n"
    f"Selected model: **{best_model_name}**\n\n"
    "### Hold-out comparison\n\n"
    f"{markdown_table(baseline_table)}\n\n"
    "Artifacts: `models/churn_v4.joblib`, `models/churn_v4_metrics.csv`.\n"
    "<!-- DAY4_RESULTS_END -->"
)

if README_PATH.exists():
    readme_text = README_PATH.read_text(encoding="utf-8")
else:
    readme_text = "# RetailMind AI\n\n"

start_marker = "<!-- DAY4_RESULTS_START -->"
end_marker = "<!-- DAY4_RESULTS_END -->"

if start_marker in readme_text and end_marker in readme_text:
    before = readme_text.split(start_marker)[0]
    after = readme_text.split(end_marker, 1)[1]
    updated_readme = before + day4_readme_section + after
else:
    updated_readme = readme_text.rstrip() + "\n\n" + day4_readme_section + "\n"

README_PATH.write_text(updated_readme, encoding="utf-8")
print("README updated:", README_PATH.resolve())
print("\n--- Day 4 README section ---\n")
print(day4_readme_section)


## 15. Final checklist

In [ ]:
checklist = {
    "Domain features engineered": "TenureGroup, billing ratios, service/household flags",
    "PCA evaluated": "5-fold CV ROC-AUC recorded",
    "Feature selection evaluated": "5-fold CV ROC-AUC recorded",
    "Bagging model evaluated": "Random Forest",
    "Boosting model evaluated": "XGBoost",
    "Bias/variance reviewed": "Train vs CV scores and learning curve",
    "Final hold-out evaluation": "Accuracy, precision, recall, F1, ROC-AUC",
    "Production pipeline saved": str(MODEL_PATH),
    "Metrics saved": str(METRICS_PATH),
    "README baseline updated": str(README_PATH)
}
for item, detail in checklist.items():
    print(f"[✓] {item}: {detail}")


## 16. Git upload

From the `RETAILMIND_AI` project root, review the changed files and commit only the Day 4 additions/updates:

```bash
git status
git add notebooks/Day_4_Feature_Engineering_Ensembles.ipynb models/churn_v4.joblib models/churn_v4_metrics.csv README.md
git commit -m "Add Day 4 feature engineering and ensemble churn model"
git push
```

Do not add the raw dataset if your repository policy excludes datasets. Do not overwrite `models/churn_v3.joblib`.

### Day 4 deliverables
- New Day 4 notebook.
- Engineered features and PCA / feature-selection comparison.
- Random Forest vs XGBoost comparison.
- Final evaluation and bias–variance review.
- `models/churn_v4.joblib` and `models/churn_v4_metrics.csv`.
- README baseline table updated.
